## Cheat Sheet for Classification

### 1. Importing Libraries
```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
```

### 2. Importing Dataset
```python
df = pd.read_csv('dataset.csv')
```

### 3. Data Preprocessing

#### 3.1. Checking for Missing Values
```python
df.isnull().sum()
```

#### 3.2. Dropping Missing Values
```python
df = df.dropna()
```

#### 3.3. Encoding Categorical Data
```python
df['column_name'] = df['column_name'].astype('category')
df['column_name'] = df['column_name'].cat.codes
```

#### 3.4. Splitting the Dataset
```python
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#### 3.5. Feature Scaling
```python
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)
```

### 4. Model Training

#### 4.1. Logistic Regression
```python
from sklearn.linear_model import LogisticRegression
classifier = LogisticRegression(random_state=42)
classifier.fit(X_train, y_train)
```

#### 4.2. K-Nearest Neighbors (KNN)
```python
from sklearn.neighbors import KNeighborsClassifier
classifier = KNeighborsClassifier(n_neighbors=5, metric='minkowski', p=2)
classifier.fit(X_train, y_train)
```

#### 4.3. Support Vector Machine (SVM)
```python
from sklearn.svm import SVC
classifier = SVC(kernel='rbf', random_state=42)
classifier.fit(X_train, y_train)
```

#### 4.4. Naive Bayes
```python
from sklearn.naive_bayes import GaussianNB
classifier = GaussianNB()
classifier.fit(X_train, y_train)
```

#### 4.5. Decision Tree
```python
from sklearn.tree import DecisionTreeClassifier
classifier = DecisionTreeClassifier(criterion='entropy', random_state=42)
classifier.fit(X_train, y_train)
```

#### 4.6. Random Forest
```python
from sklearn.ensemble import RandomForestClassifier
classifier = RandomForestClassifier(n_estimators=10, criterion='entropy', random_state=42)
classifier.fit(X_train, y_train)
```

### 5. Model Evaluation

#### 5.1. Making Predictions
```python
y_pred = classifier.predict(X_test)
```

#### 5.2. Confusion Matrix
```python
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
```

#### 5.3. Accuracy Score
```python
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, y_pred)
```

#### 5.4. Classification Report
```python
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))
```

#### 5.5. ROC Curve and AUC Score
```python
from sklearn.metrics import roc_curve, auc
fpr, tpr, thresholds = roc_curve(y_test, classifier.predict_proba(X_test)[:,1])
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()
```

### 6. Hyperparameter Tuning

#### 6.1. Grid Search
```python
from sklearn.model_selection import GridSearchCV

# Example for SVM
param_grid = {'C': [0.1, 1, 10, 100], 'gamma': [1, 0.1, 0.01, 0.001], 'kernel': ['rbf', 'poly', 'sigmoid']}
grid = GridSearchCV(SVC(), param_grid, refit=True, verbose=2)
grid.fit(X_train, y_train)

print(grid.best_params_)
print(grid.best_estimator_)
```

#### 6.2. Random Search
```python
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint as sp_randint

# Example for Random Forest
param_dist = {"max_depth": [3, None],
              "max_features": sp_randint(1, 11),
              "min_samples_split": sp_randint(2, 11),
              "min_samples_leaf": sp_randint(1, 11),
              "bootstrap": [True, False],
              "criterion": ["gini", "entropy"]}

n_iter_search = 20
random_search = RandomizedSearchCV(RandomForestClassifier(), param_distributions=param_dist, n_iter=n_iter_search)
random_search.fit(X_train, y_train)

print(random_search.best_params_)
```

### 7. Cross-Validation
```python
from sklearn.model_selection import cross_val_score

scores = cross_val_score(classifier, X, y, cv=5)
print("Accuracy: %0.2f (+/- %0.2f)" % (scores.mean(), scores.std() * 2))
```

### 8. Feature Importance (for tree-based models)
```python
importances = classifier.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure()
plt.title("Feature Importances")
plt.bar(range(X.shape[1]), importances[indices])
plt.xticks(range(X.shape[1]), [f"Feature {i}" for i in indices], rotation=90)
plt.tight_layout()
plt.show()
```

This cheat sheet covers the main steps and techniques used in classification tasks. Remember to adjust the code according to your specific dataset and requirements.


# Optuna Hyperparameter Tuning for Xgboost
```python
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 200),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
    }
    # Optuna study
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=100)

    # Best parameters
    best_params = study.best_params
    print('Best parameters:', best_params)

    # Best score
    best_score = study.best_value
    print('Best accuracy score:', best_score)

    # Create a model with the best parameters
    best_model = XGBClassifier(**best_params)
    best_model.fit(X_train, y_train)

    # Evaluate the model
    y_pred = best_model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print('Accuracy of the best model:', accuracy)

    # Plot optimization history
    optuna.visualization.plot_optimization_history(study)
    
    # Plot parameter importances
    optuna.visualization.plot_param_importances(study)

  ```


## Scaling Data & Predicting Using Random Forest
```python
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Scaling the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Training the model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

# Scaling the test data
X_test_scaled = scaler.transform(X_test)

# Predicting the test data
y_pred = model.predict(X_test_scaled)

# Evaluating the model
accuracy = accuracy_score(y_test, y_pred)
print('Accuracy:', accuracy)
```

# Predicting the test data
```python
y_pred = model.predict(X_test_scaled)
```             


## Logistic Regression with Cross Validation

```python
from sklearn import model_selection
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix,classification_report,accuracy_score,roc_auc_score
from sklearn.preprocessing import RobustScaler,MinMaxScaler,StandardScaler
acc_log=[]

kf=model_selection.StratifiedKFold(n_splits=5)
for fold , (trn_,val_) in enumerate(kf.split(X=df_nontree,y=y)):
    
    X_train=df_nontree.loc[trn_,feature_col_nontree]
    y_train=df_nontree.loc[trn_,target]
    
    X_valid=df_nontree.loc[val_,feature_col_nontree]
    y_valid=df_nontree.loc[val_,target]
    
    #print(pd.DataFrame(X_valid).head())
    ro_scaler=MinMaxScaler()
    X_train=ro_scaler.fit_transform(X_train)
    X_valid=ro_scaler.transform(X_valid)
    
    
    clf=LogisticRegression()
    clf.fit(X_train,y_train)
    y_pred=clf.predict(X_valid)
    print(f"The fold is : {fold} : ")
    print(classification_report(y_valid,y_pred))
    acc=roc_auc_score(y_valid,y_pred)
    acc_log.append(acc)
    print(f"The accuracy for Fold {fold+1} : {acc}")
    pass

### Visualizing the Decision Tree
```python
import graphviz
from sklearn import tree
# DOT data
dot_data = tree.export_graphviz(clf, out_file=None, 
                                feature_names=feature_col_tree,  
                                class_names=target,
                                filled=True)

# Draw graph
graph = graphviz.Source(dot_data, format="png") 
graph
```

### Random Forest with Cross Validation
```python
from sklearn.ensemble import RandomForestClassifier
acc_RandF=[]
kf=model_selection.StratifiedKFold(n_splits=5)
for fold , (trn_,val_) in enumerate(kf.split(X=df_tree,y=y)):
    
    X_train=df_tree.loc[trn_,feature_col_tree]
    y_train=df_tree.loc[trn_,target]
    
    X_valid=df_tree.loc[val_,feature_col_tree]
    y_valid=df_tree.loc[val_,target]
    
    clf=RandomForestClassifier(n_estimators=200,criterion="entropy")
    clf.fit(X_train,y_train)
    y_pred=clf.predict(X_valid)
    print(f"The fold is : {fold} : ")
    print(classification_report(y_valid,y_pred))
    acc=roc_auc_score(y_valid,y_pred)
    acc_RandF.append(acc)
    print(f"The accuracy for {fold+1} : {acc}")
    ```

## Checking Feature importance 
```python
plt.figure(figsize=(20,15))
importance = clf.feature_importances_
idxs = np.argsort(importance)
plt.title("Feature Importance")
plt.barh(range(len(idxs)),importance[idxs],align="center")
plt.yticks(range(len(idxs)),[feature_col_tree[i] for i in idxs])
plt.xlabel("Random Forest Feature Importance")
#plt.tight_layout()
plt.show()
```

### Measuring the performance of the model
```python
from sklearn.metrics import roc_curve, auc
fpr, tpr, thresholds = roc_curve(y_test, y_pred)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()
```

## ROC Curve Explanation
ROC (Receiver Operating Characteristic) curve is a graphical representation of the performance of a binary classifier. It plots the true positive rate (TPR) against the false positive rate (FPR) at various classification thresholds. The area under the ROC curve (AUC-ROC) is a measure of the classifier's overall performance.

- **True Positive Rate (TPR)**: Also known as sensitivity or recall, it is the proportion of actual positive cases (P) that are correctly identified by the model.
  \[
  TPR = \frac{TP}{P}
  \]
  where TP is the number of true positives. 
- **False Positive Rate (FPR)**: It is the proportion of actual negative cases (N) that are incorrectly identified as positive by the model.
  \[
  FPR = \frac{FP}{N}
  \]
  where FP is the number of false positives.
- **Threshold**: The classification threshold is the point at which the model decides whether a case is positive or negative. By varying this threshold, different points on the ROC curve are obtained.

### Interpretation of ROC Curve
- **False Positive Rate (FPR)**: It is the proportion of actual negative cases (N) that are incorrectly identified as positive by the model.    
- **True Positive Rate (TPR)**: It is the proportion of actual positive cases (P) that are correctly identified by the model.

A perfect classifier would have a TPR of 1 and an FPR of 0, resulting in a point at the top left corner of the ROC curve. In practice, most classifiers have some false positives, so the ROC curve typically starts at the origin (0,0) and moves towards the top right corner (1,1).

The area under the ROC curve (AUC-ROC) provides a single scalar summary of the model's performance. An AUC-ROC of 1 indicates a perfect classifier, while an AUC-ROC of 0.5 indicates a classifier that performs no better than random guessing.

### Importance of ROC Curve
- **Comparison of Models**: ROC curves allow for a直观比较不同分类模型（如逻辑回归、随机森林、支持向量机等）的性能。
- **Threshold Selection**: ROC曲线可以帮助选择最佳分类阈值，以平衡假阳性和假阴性。
- **Overall Performance**: AUC-ROC提供了一个单一的标量总结，可以用来比较不同模型的整体性能。
- **True Positive Rate (TPR)**: It is the proportion of actual positive cases (P) that are correctly identified by the model.
